In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split,KFold,cross_val_score
from sklearn.preprocessing import scale

In [3]:
df = pd.read_csv('dataset/wine.csv',header=None)

df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,1,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
2,1,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
3,1,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
4,1,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735


In [4]:
y = df.iloc[:,0]
X = df.iloc[:,1:]

kf = KFold(n_splits=10,shuffle=True , random_state=100)

print(f"X : {X.shape} and y : {y.shape}")

X : (178, 13) and y : (178,)


In [5]:
from sklearn.neighbors import KNeighborsClassifier

best_score = 0
best_k = None

for k in range(1,101):
    model = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(model,X,y,cv=kf,scoring='accuracy')

    mean_scoring = scores.mean()

    if mean_scoring > best_score:
        best_score = mean_scoring
        best_k = k


print(f"Лучшее качество достигается при k = {best_k}")
print(f"Максимальная точность (accuracy) = {best_score:.4f}")

Лучшее качество достигается при k = 1
Максимальная точность (accuracy) = 0.7696


In [6]:
X_scaled = scale(X)

best_acc_score = 0
best_k_score = None
sc_value = []

for k in range(1,101):
    model = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(model , X_scaled , y , cv=kf,scoring='accuracy')

    mean_scoring = scores.mean()
    sc_value.append(mean_scoring)

    if mean_scoring > best_acc_score:
        best_acc_score = mean_scoring
        best_k_score = k

print(f"Лучшее качество достигается при k = {best_k_score}")
print(f"Максимальная точность (accuracy) = {best_acc_score:.4f}")


Лучшее качество достигается при k = 31
Максимальная точность (accuracy) = 0.9833


In [11]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

best_model = KNeighborsClassifier(n_neighbors=best_k_score)

y_pred = cross_val_predict(
    best_model,
    X_scaled,
    y,
    cv=kf
)

print(classification_report(y , y_pred))

              precision    recall  f1-score   support

           1       0.97      1.00      0.98        59
           2       1.00      0.96      0.98        71
           3       0.98      1.00      0.99        48

    accuracy                           0.98       178
   macro avg       0.98      0.99      0.98       178
weighted avg       0.98      0.98      0.98       178



In [14]:
from sklearn.metrics import f1_score

f1_sco = f1_score(
    y,y_pred,
    average=None,labels=[1,2,3]
)

res = pd.DataFrame({
    'Wine Score': [1,2,3],
    'f1-score': f1_sco
})

res = res.sort_values(
    by='f1-score',
    ascending=False
)
print(res)

   Wine Score  f1-score
2           3  0.989691
0           1  0.983333
1           2  0.978417
